# Open-FDD-Style SQL Fault Rules on DuckDB `telemetry`

Prototype notebook showing how semantic `bacnet://` refs discovered via SPARQL (`pyoxigraph`) join directly with live/historical DuckDB telemetry to evaluate ASHRAE Guideline 36 / Open-FDD rules (such as **FC1: Supply Fan Running + Duct Static Pressure Below Setpoint**).


In [ ]:
from pathlib import Path
import duckdb
from live_ingest.config import load_config
from live_ingest.models import PointSample
from live_ingest.rdf_store import RdfKnowledgeBase
from live_ingest.sinks import DuckDbSink


In [ ]:
# 1. Load 223P / Brick Turtle graph into pyoxigraph and resolve bacnet:// refs
ttl_path = Path('../../examples/illustrative-ahu-refs.ttl').resolve()
cfg = load_config(ttl_path, site='demo-lab')
kb = RdfKnowledgeBase(cfg.graph)
res = kb.query('''
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ref: <https://brickschema.org/schema/Brick/ref#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?label ?ref WHERE {
  { ?p s223:hasExternalReference ?r } UNION { ?p ref:hasExternalReference ?r }
  ?r ref:BACnetURI ?ref .
  OPTIONAL { ?p rdfs:label ?label }
} ORDER BY ?ref
''')
res['rows'][:5]


In [ ]:
# 2. Seed sample telemetry into DuckDB (or connect to var/live_ingest.duckdb)
sink = DuckDbSink(':memory:')
seed = [
    PointSample('2026-11-25T12:00:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'analog-input,1', 'present-value', 0.45, 'IN_H2O', 'ok'),
    PointSample('2026-11-25T12:00:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'analog-value,1', 'present-value', 1.25, 'IN_H2O', 'ok'),
    PointSample('2026-11-25T12:00:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'binary-output,1', 'present-value', 1, '', 'ok'),
    PointSample('2026-11-25T12:05:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'analog-input,1', 'present-value', 1.22, 'IN_H2O', 'ok'),
    PointSample('2026-11-25T12:05:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'analog-value,1', 'present-value', 1.25, 'IN_H2O', 'ok'),
    PointSample('2026-11-25T12:05:00Z', 'demo-lab', 'bacnet-rpm', '10001', 'binary-output,1', 'present-value', 1, '', 'ok'),
]
sink.write(seed)


In [ ]:
# 3. Run Open-FDD FC1 SQL rule over time-bucketed telemetry keyed by bacnet:// ref
fc1_sql = '''
WITH buckets AS (
  SELECT
    time_bucket(INTERVAL '5 minutes', timestamp_utc) AS bucket_utc,
    MAX(CASE WHEN ref = 'bacnet://10001/analog-input,1' THEN value_num END) AS duct_static_pressure,
    MAX(CASE WHEN ref = 'bacnet://10001/analog-value,1' THEN value_num END) AS duct_static_sp,
    MAX(CASE WHEN ref = 'bacnet://10001/binary-output,1' THEN value_num END) AS fan_cmd
  FROM telemetry
  WHERE status = 'ok'
  GROUP BY 1
)
SELECT
  bucket_utc,
  duct_static_pressure,
  duct_static_sp,
  fan_cmd,
  (fan_cmd > 0.5 AND duct_static_pressure < (duct_static_sp - 0.15)) AS fc1_fault
FROM buckets
ORDER BY bucket_utc
'''
sink.query_sql(fc1_sql)
